# 06 Local Pilot Results

Analyze the Mac-local end-to-end FTW pilot. This is not a foundation-model result. It validates the experiment harness: real data, label budgets, adaptation modes, training loop, metrics, efficiency logging, and JSONL artifacts.

## 1. Imports

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

## 2. Load Results

In [ ]:
results_path = PROJECT_ROOT / 'artifacts' / 'local_ftw_pilot' / 'tiny_segmentation_cnn_local' / 'results.jsonl'
df = pd.read_json(results_path, lines=True)
display(df)
print('rows:', len(df))

## 3. Efficiency Summary

In [ ]:
cols = [
    'adapter',
    'label_budget',
    'num_train_examples',
    'train_seconds',
    'total_params',
    'trainable_params',
    'device',
    'memory_backend',
    'peak_vram_gb',
]
display(df[cols])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for adapter, group in df.groupby('adapter'):
    axes[0].plot(group['label_budget'], group['train_seconds'], marker='o', label=adapter)
    axes[1].plot(group['label_budget'], group['trainable_params'], marker='o', label=adapter)
axes[0].set_title('Train Seconds vs Label Budget')
axes[0].set_xlabel('Label Budget')
axes[0].set_ylabel('Seconds')
axes[1].set_title('Trainable Parameters vs Label Budget')
axes[1].set_xlabel('Label Budget')
axes[1].set_ylabel('Trainable Parameters')
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
fig.tight_layout()

## 4. Metric Summary

In [ ]:
metric_cols = [
    'adapter',
    'label_budget',
    'mean_iou',
    'foreground_iou',
    'iou_background',
    'iou_field_interior',
    'iou_field_boundary',
    'pixel_accuracy',
    'kenya_mean_iou',
    'south_africa_mean_iou',
]
display(df[metric_cols])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for adapter, group in df.groupby('adapter'):
    axes[0].plot(group['label_budget'], group['mean_iou'], marker='o', label=adapter)
    axes[1].plot(group['label_budget'], group['foreground_iou'], marker='o', label=adapter)
axes[0].set_title('Mean IoU vs Label Budget')
axes[0].set_xlabel('Label Budget')
axes[0].set_ylabel('Mean IoU')
axes[1].set_title('Foreground IoU vs Label Budget')
axes[1].set_xlabel('Label Budget')
axes[1].set_ylabel('Foreground IoU')
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
fig.tight_layout()

## 5. Label Budget Selection Audit

In [ ]:
audit = df[['adapter', 'label_budget', 'num_train_examples', 'selected_country_counts']].copy()
display(audit)

## Interpretation

This local pilot is a plumbing test, not a scientific result. The tiny CNN is too weak and the run is intentionally limited to a few train/validation batches. The useful signal is that the harness now produces comparable result rows with:

- label-budget accounting
- country-aware sample selection audit
- trainable parameter counts
- timing
- memory metadata
- per-class IoU
- per-country IoU

The next experiment step is to replace `tiny_segmentation_cnn_local` with the Prithvi feature/adapter path on GCP.